In [11]:
import numpy as np
import SI_CoRT
import CoRT_builder

import warnings
warnings.filterwarnings("ignore")

from mpmath import mp
mp.dps = 500

import json
import os
import datetime

In [12]:
import numpy as np
import CoRT_builder
import utils
from sklearn.linear_model import Lasso
from scipy.stats import norm

In [13]:
import os
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["VECLIB_MAXIMUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

In [16]:
n_target = 40
n_source = 80
p = 200
K = 5
h = 10
alpha = 0.05
T = 5
s_len = 10
s_vector = [0.5] * s_len

Ka_list = [3]
# s_vector = [0.5] * s_len
CoRT_model = CoRT_builder.CoRT()
iteration = 1000

CONST_C = 1.2
# signal_list = [0.25, 0.5, 0.75, 1]

for Ka in Ka_list:
    para_results_storage = []
    for i in range(iteration):

        if i % 50 == 0:
            print(f"Iteration {i+1}")
        target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
        X_target = target_data["X"]
        y_target = target_data["y"]
        CoRT_model = CoRT_builder.CoRT()
        similar_source_index = CoRT_model.find_similar_source(p, n_target, K, target_data, source_data, T=T, verbose=False)
        X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data)

        n = X_combined.shape[0]
        lamda = CONST_C * np.sqrt(np.log(p) / n)
        model = Lasso(alpha=lamda, fit_intercept=False, tol=1e-12, max_iter=100000)
        model.fit(X_combined, y_combined.ravel())
        beta_hat_target = model.coef_[-p:]

        M_obs = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])

        if len(M_obs) == 0:
            print("Lasso selected no features. Skipping.")
            continue
        j = np.random.choice(len(M_obs))
        selected_feature_index = M_obs[j]

        X_active, X_inactive = utils.get_active_X(beta_hat_target, X_target)

        etaj, etajTy = utils.construct_test_statistic(y_target, j, X_active)
        Sigma = np.eye(n_target)
        sigma_squared = (etaj.T @ Sigma @ etaj).item()
        sigma_z = np.sqrt(sigma_squared)
        
        naive_p_value = 2 * (1 - norm.cdf(abs(etajTy), loc = 0, scale = sigma_z))

        bonferroni_p_value = min(1.0, naive_p_value * p)

        is_signal = (selected_feature_index < s_len) 

        # if (is_signal and bonferroni_p_value <= alpha):
            # print(total_features_searched, naive_p_value)
        result_dict = {
            "p_value": bonferroni_p_value, 
            "is_signal": is_signal,
            "feature_idx": selected_feature_index
        }
        para_results_storage.append(result_dict)

    is_signal_cases = [r for r in para_results_storage if r['is_signal']]
    not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

    false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
    fpr = false_positives / len(not_signal_cases)
    true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
    tpr = true_positives / len(is_signal_cases)
    # Ví dụ: "DS", "Bonferroni", "Naive", "CoRT-SI-oc", "No inference"
    method_name = "Bonferroni" 

    # Lấy giá trị signal hiện tại (nếu s_vector là list)
    # Nếu bạn đang chạy vòng lặp 'for signal in signal_list', hãy dùng biến 'signal'
    current_signal = s_vector[0] if isinstance(s_vector, list) else s_vector

    # =========================================================
    # 2. TÍNH TOÁN KẾT QUẢ
    # =========================================================
    is_signal_cases = [r for r in para_results_storage if r['is_signal']]
    not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

    true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
    false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)

    # =========================================================
    # 3. TẠO DICTIONARY CHỨA ĐẦY ĐỦ THÔNG TIN
    # =========================================================
    stats = {
        "configs": {
            "method": method_name,
            "n_target": n_target,
            "n_source": n_source,
            "p": p,
            "alpha": alpha,
            "iteration": len(para_results_storage),
            
            # --- CÁC THAM SỐ MỚI BẠN CẦN ---
            "signal": current_signal,   # Độ mạnh tín hiệu
            "h": h,                     # Bandwidth / window size
            "Ka": Ka,                   # Số nguồn active
            "K": K,                     # Tham số K
            "T": T,                     # Threshold
            "s_len": s_len              # Độ dài tín hiệu
        },
        "metrics": {
            "fpr": false_positives / len(not_signal_cases) if len(not_signal_cases) > 0 else 0,
            "tpr": true_positives / len(is_signal_cases) if len(is_signal_cases) > 0 else 0
        },
        "raw_counts": {
            "is_signal_cases": len(is_signal_cases),
            "not_signal_cases": len(not_signal_cases),
            "true_positives": true_positives,
            "false_positives": false_positives
        }
    }

    # =========================================================
    # 4. LƯU FILE VỚI TÊN CHỨA THAM SỐ
    # =========================================================
    folder_path = 'records_other_methods'
    os.makedirs(folder_path, exist_ok=True)

    # Tên file bao gồm các tham số quan trọng để không bị ghi đè khi thay đổi cấu hình
    # Ví dụ: records/DS_nt50_sig0.5_h10_Ka2.json
    filename = (f"{folder_path}/{method_name}_"
                f"nt{n_target}_"
                f"sig{current_signal}_"
                f"h{h}_"
                f"Ka{Ka}_"
                f"iter{len(para_results_storage)}.json")

    with open(filename, 'w') as f:
        json.dump([stats], f, indent=4)

    print(f"✅ Đã lưu kết quả phương pháp [{method_name}] vào file:")
    print(f"   📂 {filename}")
    print(f"   📊 FPR: {stats['metrics']['fpr']:.4f} | TPR: {stats['metrics']['tpr']:.4f}")



Iteration 1
Iteration 51
Iteration 101
Iteration 151
Iteration 201
Iteration 251
Iteration 301
Iteration 351
Iteration 401
Iteration 451
Iteration 501
Iteration 551
Iteration 601
Iteration 651
Iteration 701
Iteration 751
Iteration 801
Iteration 851
Iteration 901
Iteration 951
✅ Đã lưu kết quả phương pháp [Bonferroni] vào file:
   📂 records_other_methods/Bonferroni_nt40_sig0.5_h10_Ka3_iter1000.json
   📊 FPR: 0.0000 | TPR: 0.0613
